# 📹 Video 3: Clustering dengan Algoritma K-Means

**Mata Kuliah**: Data Mining (21TIF604)  
**Universitas Islam Nahdlatul Ulama Jepara**  
**Dosen**: Ir. Adi Sucipto, M.Kom

---

### Tujuan Video Ini:
1. Memahami konsep algoritma K-Means untuk clustering
2. Menentukan jumlah cluster optimal dengan metode Elbow
3. Melakukan segmentasi pelanggan berdasarkan perilaku belanja
4. Mengevaluasi hasil clustering dengan Silhouette Score
5. Memvisualisasikan dan menginterpretasikan profil tiap cluster

---

### Konteks Bisnis (Renstra):
Segmentasi pelanggan membantu bisnis retail mengelompokkan pelanggan berdasarkan pola belanja, sehingga bisa memberikan perlakuan berbeda untuk setiap segmen (misal: promo khusus untuk pelanggan high-value, re-engagement untuk pelanggan pasif).

In [ ]:
# ============================================================
# STEP 1: INSTALL LIBRARY & MOUNT GOOGLE DRIVE
# ============================================================
# Google Colab sudah punya: pandas, numpy, matplotlib, seaborn, scikit-learn
# Yang perlu diinstall tambahan: mlxtend (untuk video 4, install sekalian)
# ============================================================

!pip install mlxtend -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd                           # Manipulasi data
import numpy as np                            # Operasi numerik
import matplotlib.pyplot as plt               # Visualisasi grafik
import seaborn as sns                         # Visualisasi lebih cantik
import os                                     # Operasi sistem file

from sklearn.cluster import KMeans            # Algoritma K-Means
from sklearn.preprocessing import StandardScaler  # Standarisasi fitur (mean=0, std=1)
from sklearn.metrics import silhouette_score  # Metrik evaluasi clustering
import warnings
warnings.filterwarnings('ignore')             # Sembunyikan warning agar output bersih

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)

# Folder kerja di Google Drive
DRIVE_FOLDER = '/content/drive/MyDrive/data-mining'

print("✅ Google Drive ter-mount & semua library berhasil diimport!")
print(f"📁 Folder kerja: {DRIVE_FOLDER}")

In [ ]:
# ============================================================
# STEP 2: LOAD DATA BERSIH HASIL PREPROCESSING (Video 1)
# ============================================================
# File tersimpan di Google Drive
# ============================================================

data_path = os.path.join(DRIVE_FOLDER, 'online_retail_clean.csv')
df = pd.read_csv(data_path)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"✅ Data bersih berhasil dimuat! {len(df):,} baris x {df.shape[1]} kolom")
df.head()

## Step 3: Feature Engineering — Menyiapkan Fitur untuk Clustering

Untuk clustering, kita menggunakan pendekatan **RFM** (Recency, Frequency, Monetary):
- **Recency**: seberapa baru pelanggan terakhir bertransaksi
- **Frequency**: seberapa sering pelanggan bertransaksi
- **Monetary**: berapa total uang yang dibelanjakan pelanggan

RFM adalah metode klasik dalam segmentasi pelanggan yang sangat relevan untuk konteks retail/UMKM.

In [ ]:
# ============================================================
# FEATURE ENGINEERING: RFM (Recency, Frequency, Monetary)
# ============================================================
# RFM adalah metode klasik segmentasi pelanggan di dunia retail:
#
# Recency (R) = Seberapa BARU pelanggan terakhir bertransaksi
#               Semakin kecil nilainya, semakin baru pelanggan berbelanja
#
# Frequency (F) = Seberapa SERING pelanggan bertransaksi
#                 Semakin besar, semakin sering berbelanja
#
# Monetary (M) = Berapa TOTAL uang yang dibelanjakan pelanggan
#               Semakin besar, semakin banyak uang yang dikeluarkan
# ============================================================

# Tentukan tanggal referensi = tanggal terakhir dalam dataset + 1 hari
# Ini sebagai "hari ini" untuk menghitung Recency
tanggal_referensi = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"📅 Tanggal referensi (hari ini): {tanggal_referensi.date()}")

# Kelompokkan data per CustomerID dan hitung RFM
rfm = df.groupby('CustomerID').agg(
    recency=('InvoiceDate', lambda x: (tanggal_referensi - x.max()).days),  # Hari sejak transaksi terakhir
    frequency=('InvoiceNo', 'nunique'),                                       # Jumlah transaksi unik
    monetary=('TotalAmount', 'sum')                                           # Total belanja
).reset_index()

print(f"\n✅ RFM berhasil dihitung!")
print(f"   Jumlah pelanggan: {len(rfm):,}")
print(f"\n📋 Statistik RFM:")
rfm.describe()

In [ ]:
# ============================================================
# DISTRIBUSI RFM — VISUALISASI
# ============================================================
# Melihat distribusi setiap fitur RFM untuk memahami pola data
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram Recency
axes[0].hist(rfm['recency'], bins=30, color='#e74c3c', edgecolor='white')
axes[0].set_title('Distribusi Recency (hari)', fontweight='bold')
axes[0].set_xlabel('Recency (hari sejak transaksi terakhir)')
axes[0].set_ylabel('Jumlah Pelanggan')

# Histogram Frequency
axes[1].hist(rfm['frequency'], bins=30, color='#3498db', edgecolor='white')
axes[1].set_title('Distribusi Frequency', fontweight='bold')
axes[1].set_xlabel('Frequency (jumlah transaksi)')
axes[1].set_ylabel('Jumlah Pelanggan')

# Histogram Monetary
axes[2].hist(rfm['monetary'], bins=30, color='#2ecc71', edgecolor='white')
axes[2].set_title('Distribusi Monetary', fontweight='bold')
axes[2].set_xlabel('Monetary (total belanja)')
axes[2].set_ylabel('Jumlah Pelanggan')

plt.tight_layout()
plt.show()

print("💡 Terlihat distribusi yang sangat skewed (miring). K-Means bekerja lebih baik dengan data yang terstandarisasi.")

## Step 4: Standarisasi Fitur

K-Means menggunakan **jarak Euclidean** untuk menghitung kedekatan antar data.  
Jika fitur memiliki skala yang berbeda jauh (misal: monetary dalam ribuan, frequency dalam puluhan),  
fitur dengan skala besar akan mendominasi perhitungan jarak.

**Solusi**: Standarisasi dengan `StandardScaler` → setiap fitur memiliki mean=0 dan std=1.

In [ ]:
# ============================================================
# STANDARISASI FITUR RFM
# ============================================================
# StandardScaler mengubah setiap fitur sehingga:
#   mean = 0 (rata-rata menjadi nol)
#   std = 1 (standar deviasi menjadi satu)
# Rumus: z = (x - mean) / std
#
# INI PENTING untuk K-Means karena algoritma ini menghitung jarak Euclidean.
# Tanpa standarisasi, fitur dengan skala besar (monetary) akan mendominasi.
# ============================================================

# Ambil kolom RFM saja (tanpa CustomerID)
fitur_rfm = ['recency', 'frequency', 'monetary']
X_rfm = rfm[fitur_rfm].copy()

# Standarisasi menggunakan StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)  # fit: hitung mean & std, transform: terapkan rumus z

# Konversi kembali ke DataFrame agar mudah dibaca
X_scaled_df = pd.DataFrame(X_scaled, columns=fitur_rfm)

print("✅ Standarisasi selesai!")
print(f"\n📋 Statistik setelah standarisasi (harusnya mean≈0, std≈1):")
print(X_scaled_df.describe().round(4))

## Step 5: Menentukan Jumlah Cluster Optimal — Metode Elbow

K-Means membutuhkan parameter **K** (jumlah cluster) yang ditentukan sebelumnya.  
**Metode Elbow**: jalankan K-Means dengan berbagai nilai K, lalu plot **Within-Cluster Sum of Squares (WCSS)**.  
Titik di mana penurunan WCSS mulai melambat (seperti siku/lutut) = jumlah cluster optimal.

In [ ]:
# ============================================================
# METODE ELBOW: Menentukan jumlah cluster (K) optimal
# ============================================================
# WCSS = Within-Cluster Sum of Squares (jumlah kuadrat jarak dalam cluster)
#   WCSS mengukur seberapa "rapat" data di dalam cluster-nya masing-masing
#   Semakin kecil WCSS → data semakin rapat/terkelompok dengan baik
#
# Logika Elbow:
#   - K kecil → WCSS besar (data terlalu beragam dalam satu cluster)
#   - K besar → WCSS kecil (setiap data punya cluster sendiri, tidak berguna)
#   - K optimal = titik di mana penurunan WCSS mulai melambat (seperti siku/lutut)
# ============================================================

# Coba K dari 1 sampai 10
k_range = range(1, 11)
wcss = []  # List untuk menyimpan nilai WCSS tiap K

for k in k_range:
    # Inisialisasi KMeans dengan K cluster
    # n_init=10 = jalankan 10 kali, pilih hasil terbaik (menghindari local minimum)
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)  # inertia_ = nilai WCSS

# Visualisasi Elbow Curve
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_range, wcss, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Jumlah Cluster (K)', fontsize=13)
ax.set_ylabel('WCSS (Within-Cluster Sum of Squares)', fontsize=13)
ax.set_title('Metode Elbow — Menentukan Jumlah Cluster Optimal', fontsize=15, fontweight='bold')
ax.set_xticks(list(k_range))
ax.grid(True, alpha=0.3)

# Tandai area "elbow" (perkiraan)
ax.axvspan(3, 5, alpha=0.15, color='red', label='Area Elbow (K=3-5)')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("💡 Cari titik di mana garis mulai 'melandai' (seperti siku/lutut).")
print("   Dari grafik, K=3 atau K=4 tampak sebagai titik elbow.")

In [ ]:
# ============================================================
# SILHOUETTE SCORE: Konfirmasi jumlah cluster optimal
# ============================================================
# Silhouette Score mengukur seberapa baik data dikelompokkan:
#   Rentang: -1 sampai +1
#   +1 = data sangat cocok di cluster-nya, jauh dari cluster lain (ideal)
#    0 = data berada di batas antar cluster (ambigu)
#   -1 = data mungkin salah cluster
#
# Kita hitung silhouette score untuk K=2 sampai K=7
# K dengan silhouette score TERTINGGI = jumlah cluster optimal
# ============================================================

k_range_sil = range(2, 8)  # Silhouette score butuh minimal 2 cluster
silhouette_scores = []

for k in k_range_sil:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"   K={k} → Silhouette Score = {score:.4f}")

# Visualisasi Silhouette Score
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_range_sil, silhouette_scores, 'go-', linewidth=2, markersize=10)
ax.set_xlabel('Jumlah Cluster (K)', fontsize=13)
ax.set_ylabel('Silhouette Score', fontsize=13)
ax.set_title('Silhouette Score — Konfirmasi Jumlah Cluster Optimal', fontsize=15, fontweight='bold')
ax.set_xticks(list(k_range_sil))
ax.grid(True, alpha=0.3)

# Tandai K terbaik
best_k = list(k_range_sil)[np.argmax(silhouette_scores)]
ax.axvline(x=best_k, color='red', linestyle='--', label=f'K terbaik = {best_k}')
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

print(f"\n✅ Jumlah cluster optimal berdasarkan Silhouette Score: K={best_k}")

## Step 6: Menjalankan K-Means dengan K Optimal

Sekarang kita jalankan K-Means dengan jumlah cluster optimal yang sudah ditentukan.  
Kita juga akan memberikan **nama segmen** untuk setiap cluster berdasarkan karakteristiknya.

In [ ]:
# ============================================================
# MENJALANKAN K-MEANS DENGAN K OPTIMAL
# ============================================================
# Kita gunakan K=best_k (ditentukan dari silhouette score)
# init='k-means++' = metode inisialisasi cerdas yang memilih centroid awal
#   secara strategis agar konvergen lebih cepat dan hasil lebih baik
# n_init=10 = jalankan 10 kali, pilih hasil WCSS terkecil
# random_state=42 = seed untuk reproducibility
# ============================================================

k_optimal = best_k  # Gunakan K terbaik dari silhouette score

kmeans_final = KMeans(
    n_clusters=k_optimal,       # Jumlah cluster
    init='k-means++',           # Inisialisasi cerdas
    n_init=10,                  # Ulangi 10 kali, pilih terbaik
    random_state=42,            # Seed untuk hasil konsisten
    verbose=0                   # Jangan tampilkan log detail
)

# Latih model dan prediksi cluster untuk setiap pelanggan
cluster_labels = kmeans_final.fit_predict(X_scaled)

# Tambahkan label cluster ke dataframe RFM
rfm['Cluster'] = cluster_labels

# Hitung jumlah pelanggan per cluster
print(f"✅ K-Means berhasil dijalankan dengan K={k_optimal}!")
print(f"   Jumlah iterasi konvergen: {kmeans_final.n_iter_}")
print(f"\n📋 Jumlah pelanggan per cluster:")
for c in range(k_optimal):
    count = (rfm['Cluster'] == c).sum()
    print(f"   Cluster {c}: {count} pelanggan ({count/len(rfm)*100:.1f}%)")

In [ ]:
# ============================================================
# PROFIL CLUSTER: Karakteristik rata-rata setiap cluster
# ============================================================
# Kita hitung rata-rata RFM per cluster untuk memahami profil tiap segmen
# Ini penting untuk memberikan NAMA segmen yang bermakna
# ============================================================

# Hitung rata-rata RFM per cluster
cluster_profile = rfm.groupby('Cluster')[fitur_rfm].mean().round(2)
cluster_profile['Jumlah_Pelanggan'] = rfm.groupby('Cluster')['CustomerID'].count()

print("📋 Profil Rata-Rata Setiap Cluster:")
print("=" * 70)
print(cluster_profile)

# Berikan nama segmen berdasarkan karakteristik
# Logika penamaan:
#   - Recency RENDAH + Frequency TINGGI + Monetary TINGGI = Pelanggan Premium
#   - Recency SEDANG + Frequency SEDANG + Monetary SEDANG = Pelanggan Reguler
#   - Recency TINGGI + Frequency RENDAH + Monetary RENDAH = Pelanggan Pasif
print("\n💡 Interpretasi Cluster:")
for c in range(k_optimal):
    row = cluster_profile.loc[c]
    print(f"\n   Cluster {c}:")
    print(f"   - Recency rata-rata: {row['recency']:.0f} hari")
    print(f"   - Frequency rata-rata: {row['frequency']:.0f} transaksi")
    print(f"   - Monetary rata-rata: £{row['monetary']:.0f}")
    print(f"   - Jumlah pelanggan: {int(row['Jumlah_Pelanggan'])}")

In [ ]:
# ============================================================
# VISUALISASI CLUSTER: Scatter plot 2D (Recency vs Monetary)
# ============================================================
# Kita visualisasikan cluster dalam 2 dimensi untuk melihat pemisahan
# Warna berbeda = cluster berbeda
# ============================================================

fig, ax = plt.subplots(figsize=(10, 7))

# Warna untuk setiap cluster
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for c in range(k_optimal):
    mask = rfm['Cluster'] == c
    ax.scatter(
        rfm.loc[mask, 'recency'],
        rfm.loc[mask, 'monetary'],
        c=colors[c % len(colors)],
        label=f'Cluster {c}',
        alpha=0.6,
        s=30
    )

# Tandai posisi centroid (pusat cluster) dalam skala asli
centroids_scaled = kmeans_final.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

for c in range(k_optimal):
    ax.scatter(
        centroids_original[c][0],  # Recency centroid
        centroids_original[c][2],  # Monetary centroid
        marker='X', s=200, c=colors[c % len(colors)],
        edgecolors='black', linewidths=2,
        label=f'Centroid {c}'
    )

ax.set_xlabel('Recency (hari sejak transaksi terakhir)', fontsize=12)
ax.set_ylabel('Monetary (total belanja £)', fontsize=12)
ax.set_title('Segmentasi Pelanggan — K-Means Clustering', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALISASI: Pairplot RFM per Cluster
# ============================================================
# Pairplot menampilkan scatter plot semua kombinasi pasangan fitur
# Diagonal = distribusi (histogram/KDE) per cluster
# Ini membantu melihat bagaimana cluster terpisah di berbagai dimensi
# ============================================================

# Tambahkan kolom Cluster sebagai kategori untuk warna
rfm_plot = rfm.copy()
rfm_plot['Cluster'] = rfm_plot['Cluster'].astype(str)

# Buat pairplot
g = sns.pairplot(
    rfm_plot,
    vars=fitur_rfm,
    hue='Cluster',
    palette=colors[:k_optimal],
    diag_kind='kde',        # Diagonal: Kernel Density Estimation (kurva halus)
    plot_kws={'alpha': 0.5, 's': 20},
    diag_kws={'alpha': 0.7}
)

g.fig.suptitle('Pairplot RFM per Cluster — Segmentasi Pelanggan', 
               fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EVALUASI CLUSTERING: SILHOUETTE SCORE FINAL
# ============================================================
# Silhouette Score akhir untuk model yang sudah dipilih
# Rentang: -1 (buruk) sampai +1 (sangat baik)
# ============================================================

final_score = silhouette_score(X_scaled, cluster_labels)
print(f"🎯 Silhouette Score (K={k_optimal}): {final_score:.4f}")
print()
print("📋 Panduan interpretasi Silhouette Score:")
print("   0.71 - 1.00 → Struktur cluster sangat kuat")
print("   0.51 - 0.70 → Struktur cluster cukup baik")
print("   0.26 - 0.50 → Struktur cluster lemah (tapi masih bisa diterima)")
print("   < 0.25      → Tidak ada struktur cluster yang jelas")

In [ ]:
# ============================================================
# INTERPRETASI HASIL & REKOMENDASI BISNIS
# ============================================================
# Penamaan segmen berdasarkan PERINGKAT ANTAR CLUSTER
# (bukan sekadar di atas/bawah median keseluruhan)
# Ini lebih akurat karena membandingkan cluster satu sama lain
# ============================================================

# Urutkan cluster berdasarkan Monetary (dari tertinggi) untuk peringkat
cluster_ranked = cluster_profile.sort_values('monetary', ascending=False)

# Tentukan nama segmen berdasarkan posisi relatif antar cluster
# Logika: bandingkan karakteristik cluster terhadap rata-rata keseluruhan
overall_mean = rfm[fitur_rfm].mean()

print("=" * 70)
print("📊 INTERPRETASI HASIL CLUSTERING K-MEANS")
print("=" * 70)
print()
print(f"1. JUMLAH CLUSTER OPTIMAL: K={k_optimal}")
print(f"   → Ditentukan dari Elbow Method & dikonfirmasi Silhouette Score")
print()
print(f"2. SILHOUETTE SCORE: {final_score:.4f}")
print(f"   → Menunjukkan seberapa baik pelanggan terkelompokkan")
print()
print("3. REKOMENDASI STRATEGI PER SEGMENT:")
print()

# Mapping cluster ke nama segmen berdasarkan karakteristik relatif
segment_names = {}

for c in range(k_optimal):
    row = cluster_profile.loc[c]
    r, f, m = row['recency'], row['frequency'], row['monetary']

    # Logika penamaan berdasarkan kombinasi RFM:
    # - Frequency & Monetary TINGGI + Recency RENDAH = Premium (belanja besar & sering, masih aktif)
    # - Frequency & Monetary SEDANG + Recency RENDAH = Reguler (belanja normal, masih aktif)
    # - Recency TINGGI + Monetary TINGGI = Berisiko (pernah belanja besar, tapi sudah lama tidak)
    # - Recency TINGGI + Monetary RENDAH = Pasif (jarang belanja & sudah lama tidak)

    # Cek apakah frequency & monetary di atas rata-rata keseluruhan
    freq_tinggi = f > overall_mean['frequency']
    mon_tinggi = m > overall_mean['monetary']
    rec_tinggi = r > overall_mean['recency']

    if freq_tinggi and mon_tinggi and not rec_tinggi:
        nama = "PELANGGAN PREMIUM"
        strategi = "Program loyalty VIP, undangan event eksklusif, promo personal, layanan prioritas"
    elif not rec_tinggi and not mon_tinggi:
        nama = "PELANGGAN REGULER"
        strategi = "Up-sell produk premium, bundling, promo reguler untuk tingkatkan spending"
    elif rec_tinggi and mon_tinggi:
        nama = "PELANGGAN BERISIKO TINGGI"
        strategi = "Re-engagement segera! Diskon khusus, reminder bahwa mereka pernah loyal belanja besar"
    else:
        nama = "PELANGGAN PASIF"
        strategi = "Email re-engagement, diskon besar, reminder produk, kampanye awareness"

    segment_names[c] = nama
    print(f"   Cluster {c} → {nama}")
    print(f"   - Recency={r:.0f} hari, Frequency={f:.0f}x, Monetary=£{m:.0f}")
    print(f"   - Jumlah: {int(row['Jumlah_Pelanggan'])} pelanggan ({int(row['Jumlah_Pelanggan'])/len(rfm)*100:.1f}%)")
    print(f"   - Strategi: {strategi}")
    print()

print("4. KELEBIHAN K-MEANS:")
print("   - Cepat dan efisien untuk dataset besar")
print("   - Mudah diimplementasikan dan diinterpretasi")
print("   - Cocok untuk segmentasi pelanggan (RFM)")
print()
print("5. KEKURANGAN K-MEANS:")
print("   - Harus menentukan K terlebih dahulu")
print("   - Sensitif terhadap outlier (sudah kita tangani di preprocessing)")
print("   - Asumsi cluster berbentuk spherical (bulat)")
print("   - Bisa terjebak di local minimum → gunakan n_init > 1")